In [1]:
from __future__ import annotations

import random
import heapq

convert_hour_to_second = lambda hour: hour * 60 * 60

In [3]:
class ConsultationRequest:
    def __init__(self, arrival_time, service_time):
        self.arrival_time: float = arrival_time
        self.service_time: float = service_time

    def __lt__(self, other: ConsultationRequest) -> bool:
        return self.arrival_time < other.arrival_time

    @classmethod
    def generate_requests(
            cls, arrival_distribution: list[float], service_distribution: list[int],
            num_requests: int, start_hour: int
    ) -> list[ConsultationRequest]:
        requests = []
        current_time = convert_hour_to_second(start_hour)

        for _ in range(num_requests):
            arrival_time = random.choice(arrival_distribution)
            service_time = random.choice(service_distribution)
            current_time += arrival_time
            requests.append(cls(current_time, service_time))

        return requests


class Consultant:
    def __init__(self, consultant_id: int):
        self.consultant_id: int = consultant_id
        self.is_busy: bool = False
        self.end_time: int = 0

    def assign_request(self, request: ConsultationRequest, current_time: int) -> bool:
        if not self.is_busy:
            self.is_busy = True
            self.end_time = current_time + request.service_time
            return True
        return False

    def release_consultant(self, current_time: int):
        if self.is_busy and self.end_time <= current_time:
            self.is_busy = False

In [4]:
class VirtualNode:
    def __init__(self, num_consultants: int, start_working_hour: int):
        self.consultants: list[Consultant] = [Consultant(i) for i in range(num_consultants)]
        self.waiting_queue: list[ConsultationRequest] = []
        self.current_time: int = convert_hour_to_second(start_working_hour)

    def assign_request(self, request: ConsultationRequest) -> bool:
        return any(consultant.assign_request(request, current_time=self.current_time) for consultant in self.consultants)

    def release_consultants(self) -> None:
        for consultant in self.consultants:
            consultant.release_consultant(current_time=self.current_time)

    def simulate(self, requests: list[ConsultationRequest]) -> tuple[int, int, int]:
        waiting_times = []

        for request in requests:
            self.current_time = request.arrival_time
            self.release_consultants()

            if not self.assign_request(request):
                heapq.heappush(self.waiting_queue, request)

            while self.waiting_queue:
                self.release_consultants()
                next_request = heapq.heappop(self.waiting_queue)
                if self.assign_request(next_request):
                    waiting_times.append(self.current_time - next_request.arrival_time)
                else:
                    heapq.heappush(self.waiting_queue, next_request)
                    break

        self.current_time = max(c.end_time for c in self.consultants)
        closed_time = self.current_time

        avg_waiting_time = sum(waiting_times) / len(waiting_times) if waiting_times else 0
        return len(waiting_times), avg_waiting_time, closed_time

In [5]:
a = (1, 2)
start_time = 12  # K - початок надходження запитів (о 12 годині) 
end_time = start_time + a[0]  # K + h - кінець надходження запитів (о K + a годині)
m = 12  # M - початок обробки запитів 
n = start_time + a[0]  # N - кількість консультантів

print(f"Діапазон надходження запитів з {start_time} по {end_time} годин.")
print(f"Початок обробки запитів з {m} годин.")
print(f"Кількість консультантів: {n}.")

Діапазон надходження запитів з 12 по 13 годин.
Початок обробки запитів з 12 годин.
Кількість консультантів: 13.


In [6]:
if __name__ == "__main__":
    random_requests = ConsultationRequest.generate_requests(
        arrival_distribution=[0.1, 0.2, 0.3, 0.4, 0.5], service_distribution=[1, 2, 3, 4, 5],
        num_requests=10000, start_hour=start_time
    )
    node = VirtualNode(num_consultants=n, start_working_hour=m)
    result = node.simulate(random_requests)

    print("Результати симуляції:")
    print(f"Кількість запитів, що очікували в черзі: {result[0]}")
    print(f"Середній час очікування: {result[1]:.2f} сек.")
    print(f"Час завершення роботи вузла: {result[2] / 60 / 60:.2f} сек.")

Результати симуляції:
Кількість запитів, що очікували в черзі: 835
Середній час очікування: 1.24 сек.
Час завершення роботи вузла: 12.82 сек.
